# 🐍 Python for Chemists — Day 4
## Photocatalytic dye degradation: C/C₀ plot + rate constant (k) in one click

**What this notebook does**
1. Reads your UV-Vis absorbance values (at the dye's λmax, e.g. 664 nm for methylene blue)
2. Converts absorbance to C/C₀ (Beer–Lambert: C/C₀ = A/A₀)
3. Plots the degradation curve (dark period shaded)
4. Fits pseudo-first-order kinetics: ln(C₀/C) = k·t
5. Gives a table of **k, R², half-life and % removal** for every sample
6. Checks that the k ranking matches the % removal ranking

> 📌 The data file `MB_degradation_absorbance.csv` is **simulated** for practice. Replace it with your own file (same layout: first column = time in min, one column per sample).

*Muhammad Amjad — Python for Chemists series*

### Step 1 — Load the data
First column: time (min). Negative time = dark (adsorption) period. **t = 0 is the moment the light is switched on.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

# In Google Colab: upload your file with the folder icon on the left, then put its name here
FILE = "MB_degradation_absorbance.csv"

df = pd.read_csv(FILE)
time_col = df.columns[0]
samples = df.columns[1:]
df

### Step 2 — Absorbance → C/C₀
Beer–Lambert says absorbance is proportional to concentration, so **C/C₀ = A/A₀**. You don't need a calibration curve for C/C₀.

**Which C₀?** For kinetics we use the concentration **at t = 0 (after dark adsorption–desorption equilibrium)**. The dye removed in the dark was adsorbed, not degraded, so it must not be counted in k.

In [ ]:
light = df[df[time_col] >= 0].reset_index(drop=True)   # illuminated part only
C0 = light.iloc[0][samples]                               # absorbance at t = 0 (light on)

CC0_all = df[samples] / C0          # C/C0 for the full experiment (dark points > 1 is normal: before adsorption)
CC0_light = light[samples] / C0
CC0_light.insert(0, time_col, light[time_col])
CC0_light.round(3)

### Step 3 — Degradation curve

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
markers = ["o", "s", "^", "D", "v", "P"]
for i, s in enumerate(samples):
    ax.plot(df[time_col], CC0_all[s], marker=markers[i % 6], label=s)
ax.axvspan(df[time_col].min(), 0, color="grey", alpha=0.15)
ax.text(df[time_col].min() / 2, 0.05, "Dark", ha="center")
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Time (min)")
ax.set_ylabel("C/C$_0$")
ax.set_title("Photocatalytic degradation")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Step 4 — Pseudo-first-order kinetics
If the plot of **ln(C₀/C) vs t** is a straight line, the slope is the rate constant **k (min⁻¹)**. R² tells you how well the straight line fits.

In [ ]:
results = []
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, s in enumerate(samples):
    t = light[time_col].values
    y = np.log(C0[s] / light[s].values)
    fit = linregress(t, y)
    ax.plot(t, y, markers[i % 6], label=f"{s}: k = {fit.slope:.4f} min$^{{-1}}$, R$^2$ = {fit.rvalue**2:.3f}")
    ax.plot(t, fit.intercept + fit.slope * t, "-", color=ax.lines[-1].get_color())
    removal = (1 - light[s].values[-1] / C0[s]) * 100
    results.append({"Sample": s,
                    "k (min⁻¹)": round(fit.slope, 4),
                    "R²": round(fit.rvalue**2, 3),
                    "t½ (min)": round(np.log(2) / fit.slope, 1),
                    f"% removal at {t[-1]:.0f} min": round(removal, 1)})
ax.set_xlabel("Time (min)")
ax.set_ylabel("ln(C$_0$/C)")
ax.set_title("Pseudo-first-order fit")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

table = pd.DataFrame(results)
table

### Step 5 — Sanity check
A faster catalyst (bigger k) should also remove more dye. If the two rankings disagree, check your data before writing the paper.

In [ ]:
rank_k = table.sort_values("k (min⁻¹)", ascending=False)["Sample"].tolist()
rank_removal = table.sort_values(table.columns[-1], ascending=False)["Sample"].tolist()
print("Ranking by k       :", rank_k)
print("Ranking by % removal:", rank_removal)
print("✅ Rankings match" if rank_k == rank_removal else "⚠️ Rankings do NOT match — check your data!")
low_r2 = table[table["R²"] < 0.95]["Sample"].tolist()
if low_r2:
    print("⚠️ R² below 0.95 for:", low_r2, "→ pseudo-first-order may not describe these well.")

### Step 6 — Save results
Saves the results table as a CSV file you can open in Excel.

In [ ]:
table.to_csv("kinetics_results.csv", index=False)
print("Saved kinetics_results.csv")

### ⚠️ Common mistakes
1. Using the concentration **before** the dark period as C₀ for k → adsorption gets counted as degradation.
2. Reporting k **without R²**.
3. Comparing k values measured under **different** light sources, catalyst doses, dye concentrations or pH.
4. A k ranking that doesn't match the % removal ranking (usually a typing or data error).

*Next: try it with your own absorbance data!*